## 0. Preparación del entorno

In [ ]:
# Importamos las librerías necesarias

import getpass
import json
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Literal

from pathlib import Path
from pypdf import PdfReader

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from jsonschema import Draft202012Validator
from openai import OpenAI

In [ ]:
# Configuramos la conexión con el LLM de OpenAI

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)


if not os.getenv("OPENAI_API_KEY"):
    print("La API Key no ha sido encontrada en el archivo .env.")
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce la OpenAI API key: ")

client = OpenAI()

#Configuramos un modelo de generación de respuestas y otro para hacer el embedding

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

# Creamos un diccionario con los parámetros de configuración del LLM.

# El valor de temperature se mantiene bajo para basarse más estrictamente en la información de TENERIFE.pdf. Como estamos configurando explicitamente temperature entonces no es conveniente modificar top_p.

LLM_PARAMS = {
    "model": GENERATION_MODEL,
    "temperature": 0.2,       
    "max_tokens": 800      
}


print(f"Modelo de generación: {GENERATION_MODEL}")
print(f"Modelo de embeddings: {EMBEDDING_MODEL}")

Modelo de generación: gpt-4.1-mini
Modelo de embeddings: text-embedding-3-small


## 1. Preparación de la base documental

In [ ]:


def cargar_documento_tenerife() -> list[dict[str, str]]:
    """Carga el documento TENERIFE.pdf desde la carpeta data."""
    
    file_path = Path("../data/TENERIFE.pdf")
    
    # Validación en caso de que no esté presente el pdf de Tenerife
    if not file_path.exists():
        raise FileNotFoundError(f"¡Error! No se encontró el archivo. Ruta buscada: {file_path.resolve()}")
        
    # Extracción del texto del PDF
    extracted_text = ""
    reader = PdfReader(file_path)
    
    for page in reader.pages:
        extracted_text += page.extract_text() + "\n"
        
    return [
        {
            "source": file_path.name,
            "text": extracted_text.strip()
        }
    ]

try:
    raw_docs = cargar_documento_tenerife()
    print(f"Documentos cargados: {len(raw_docs)}")
    print(f"- Archivo: {raw_docs[0]['source']} | Caracteres extraídos: {len(raw_docs[0]['text'])}")
except Exception as e:
    print(e)

Documentos cargados: 1
- Archivo: TENERIFE.pdf | Caracteres extraídos: 16202


## 2. Chunking

In [18]:
def chunk_text(text: str, *, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """Divide texto en chunks con solapamiento por caracteres, evitando bucles infinitos."""
    cleaned = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    if len(cleaned) <= chunk_size:
        return [cleaned]

    chunks = []
    start = 0
    while start < len(cleaned):
        end = min(start + chunk_size, len(cleaned))
        chunks.append(cleaned[start:end])
        if end == len(cleaned):
            break
        start = max(0, end - overlap)

    return chunks


chunks: list[dict[str, Any]] = []
for doc in raw_docs:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        chunks.append(
            {
                "id": f"{doc['source']}::chunk_{i}",
                "source": doc["source"],
                "chunk_index": i,
                "text": chunk,
            }
        )

print("Chunks generados:", len(chunks))
print(chunks[0]["id"])
print(chunks[0]["text"][:1000])



Chunks generados: 20
TENERIFE.pdf::chunk_0
TENERIFE – LUGARES DE INTERÉS
SITIOS QUE VER
ZONA NORTE
• Santa Cruz de Tenerife:
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa
Cruz sería:
- Aparcar en el aparcamiento del Parque Marítimo (ubicación).
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación).
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación).
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo
dirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria -
ubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del
Príncipe - ubicación).
- Volver de nuevo a por el coche.
- Ir hacia la playa de Las Teresitas (ubicación).
Los puntos de interés sugeridos son estos:
o Parque Marítimo César Manrique [vídeo - ubicación]
Conjunto de piscinas naturales.
o Auditorio de Tenerife [vídeo - ubicación]
o Plaza de España [vídeo - ubicación]
o Parque Garcí

## 3. Embeddings

In [19]:
def embed_texts(texts: list[str], *, model: str = EMBEDDING_MODEL) -> np.ndarray:
    """Genera embeddings para una lista de textos y devuelve una matriz numpy."""
    response = client.embeddings.create(model=model, input=texts)
    vectors = [item.embedding for item in response.data]
    return np.array(vectors, dtype=np.float32)


def cosine_similarity_matrix(query_vector: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Calcula similitud coseno entre un vector y cada fila de una matriz."""
    query_norm = np.linalg.norm(query_vector)
    matrix_norms = np.linalg.norm(matrix, axis=1)
    denominator = np.maximum(query_norm * matrix_norms, 1e-12)
    return (matrix @ query_vector) / denominator


chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = embed_texts(chunk_texts)

print("Shape embeddings:", chunk_embeddings.shape)

Shape embeddings: (20, 1536)
